In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

from sklearn.metrics import mean_squared_error

import os
import itertools
import warnings
from joblib import Parallel, delayed

!pip install pytrends
from pytrends.request import TrendReq

warnings.filterwarnings("ignore")

# **Exploratory Data Analysis**
## **1. Visualizations**

In [ ]:
#Visualization Part 1
datasets = ["CAICLAIMS", "NYICLAIMS", "INICLAIMS", "HIICLAIMS"]

state_names = {
    "CAICLAIMS": "California",
    "NYICLAIMS": "New York",
    "INICLAIMS": "Indiana",
    "HIICLAIMS": "Hawaii"
}

claims_dfs = {}

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)

for ax, code in zip(axes, datasets):
    df = pd.read_csv(f"Data/{code}.csv")
    df.rename(columns={code: "Claims"}, inplace=True)
    df["observation_date"] = pd.to_datetime(df["observation_date"])
    df = df.set_index("observation_date")
    claims_dfs[code] = df
    ax.plot(df.index, df["Claims"])
    ax.set_title(f"{code}: Initial Claims Filed in {state_names[code]}")
    ax.set_ylabel("Number of Claims")

first_df = claims_dfs[datasets[0]]
years = first_df.index.year.unique()
years = years[::5]

tick_positions = pd.to_datetime(years, format="%Y")

axes[-1].set_xticks(tick_positions)
axes[-1].set_xticklabels(years)
axes[-1].set_xlabel("Year")

fig.suptitle("Initial Unemployment Insurance Claims by State", fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
#Visualization Part 2
recessions = pd.read_csv("Data/JHDUSRGDPBR.csv")
recessions["observation_date"] = pd.to_datetime(recessions["observation_date"])
recessions = recessions.set_index("observation_date")
start_date = min(claims_dfs[code].index.min() for code in datasets)
end_date = max(claims_dfs[code].index.max() for code in datasets)
recessions = recessions.loc[start_date:end_date]
#Cut off some months in recession data to fit the time period of Unemployment Insurance Claims across the 4 states

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for code in datasets:
    df = claims_dfs[code]
    ax.plot(df["Claims"], label=state_names[code])

ax.fill_between(
    recessions.index,
    0,
    1,
    where=recessions["JHDUSRGDPBR"] == 1,
    color="gray",
    alpha=0.3,
    transform=ax.get_xaxis_transform(),
    label="US Recession"
)

ax.set_title("Initial Unemployment Insurance Claims by State")
ax.set_xlabel("Date")
ax.set_ylabel("Number of Claims")
ax.legend()
plt.tight_layout()
plt.show()

Let's shade the recession periods and plot data from all 4 states. The recession shading data is from FRED, in a dataset called JHDUSRGDPBR. California has the highest number of unemployment claims, during recessions and during non-recession periods. This is likely due to California having the largest population compared to IN, NY, and HI. 

The number of claims shows a slightly upward trend during recessions (shaded areas), with the pandemic showing a huge spike in number of unemployment insurance claims filed.

## **2. A Basic Overview Of CAICLAIMS, NYICLAIMS, INICLAIMS, HIICLAIMS**

Let's make a dataframe that contains the initial unemployment insurance claim for all states.

In [ ]:
df_all = pd.concat([claims_dfs[code]["Claims"] for code in datasets],axis=1)
df_all.columns = ["CA", "NY", "IN", "HI"]
df_all

In [ ]:
df_all.describe()

In [ ]:
df_all.plot(kind="box", figsize=(8,10))
plt.title("Distribution of Weekly Unemployment Claims by State")
plt.ylabel("Number of Claims")
plt.xlabel("State")
plt.show()

The summary statistics for df_all show the distribution of weekly unemployment claims by state. California has the highest median and maximum number of unemployment claims, and also has the widest spread and the highest outliers. But California's large number of unemployment claims makes it difficult to see the plots for other graphs. Let's graph these plots on a log scale.

In [ ]:
df_all_log = np.log1p(df_all)
df_all_log.plot(kind="box", figsize=(8,6))

plt.title("Log-Scaled Distribution of Claims by State")
plt.ylabel("log(Claims + 1)")
plt.show()

California dominates due to much higher absolute claim counts, but the log scale reveals underlying distribution patterns more clearly. Some states show slightly wider boxes and longer whiskers, showing greater relative volatility in claims. Outliers still remain after log-scaling, highlighting extreme economic (ex. COVID-19 pandemic).

In [ ]:
df_all.dtypes

The number of claims for each state is a float, which means it can be easily manipulated for computations (and is a good thing!)

In [ ]:
df_all.isna().any()

The datasets for CA and HI have some NaNs in the column for the number of claims, which we may have to deal with later.

## **3. State-level Google Trends data for unemployment-related search terms**

Generate Google Trends data and cache to CSVs for reproducibility. The Google Trends dataset will begin in 2004 and end in 2026. Although our dataset for unemployment claims begins at around 1986, the earliest available Google Trends data begins in 2004.

In [ ]:
pytrends = TrendReq(hl='en-US', tz=360)
keywords = [
    "unemployment",
    "unemployment benefits",
    "file for unemployment",
    "unemployment office",
    "apply for unemployment",
]

states = {
    "CA": "US-CA",
    "HI": "US-HI",
    "NY": "US-NY",
    "IN": "US-IN"
}

for state, geo in states.items():
    pytrends.build_payload(
        kw_list=keywords,
        timeframe='2004-01-01 2026-04-01',  # earliest usable data
        geo=geo
    )
    
    df = pytrends.interest_over_time()
    df = df.drop(columns=['isPartial'])
    
    df.index = pd.to_datetime(df.index)
    df = df.resample('W').mean()
    
    df.to_csv(f"google_trends_{state}.csv")

Load in CSV and remove NaNs from the dataset. NaNs are weeks with no Google Trends data on unemployment search terms.

In [ ]:
states = ["CA", "HI", "NY", "IN"]
google_trends = {}  # dictionary of datasets

for s in states:
    df = pd.read_csv(f"google_trends_{s}.csv")
    
    if "date" not in df.columns:
        df = df.rename(columns={"Unnamed: 0": "date"})
    
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date")
    
    google_trends[s] = df.dropna()

Make a visualization for Google Trends unemployment keywords search in California, Hawaii, New York, and Indiana.

In [ ]:

for s, df in google_trends.items():
    fig, ax = plt.subplots(figsize=(12, 6))
    
    avg_trend = df.mean(axis=1)
    
    ax.scatter(df.index, avg_trend, s=10, alpha=0.3)
    avg_trend.rolling(4).mean().plot(ax=ax, linewidth=2)
    
    ax.set_title(f"{s} Google Trends (Scatter + Trend)")
    ax.set_ylabel("Search Interest")
    
    plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

trend_mean.plot(ax=ax, title="Average Unemployment Search Interest by State")
ax.set_ylabel("Search Interest")

plt.show()

# **CAICLAIMS Model Testing**

## **1. ARIMA Baseline Model**

In [ ]:
y_CA = claims_dfs["CAICLAIMS"]["Claims"]

test_horizon = 104  # final 2 years
train_end = len(y_CA) - test_horizon
ttrain = np.arange(train_end)
ttest = np.arange(train_end, len(y_CA))
tpred = np.arange(train_end, len(y_CA) + test_horizon)  # future time points for forecasting, 2 years ahead

ytrain_CA = y_CA.iloc[:train_end]
ytest_CA = y_CA.iloc[train_end:]

plt.figure(figsize=(10, 6))
plt.plot(ttrain, ytrain_CA, label="Training Data", color="steelblue")
plt.plot(ttest, ytest_CA, label="Testing Data", color="red")
plt.axvline(x=train_end, color="gray", linestyle="--", label="Train/Test Split")
plt.title("CAICLAIMS: Initial Claims Filed in California")
plt.xlabel("Time (Weekly: Feb 8, 1986 - April 11, 2026)")
plt.ylabel("Number of claims")
plt.legend(); plt.tight_layout(); plt.show();

### **Training Data Time Series Analysis**

In [ ]:
p_max = 50
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6))
plot_acf(ytrain_CA.dropna(), lags=p_max, ax=ax1, title="Sample ACF")
plot_pacf(ytrain_CA.dropna(), lags=p_max, ax=ax2, title="Sample PACF")
plt.show();

In [ ]:
ytrain_CA_diff = ytrain_CA.diff()

plt.figure(figsize=(10,6))
plt.plot(ytrain_CA_diff.dropna())
plt.xlabel("Time (Weekly: Feb 8, 1986 - April 11, 2026)")
plt.ylabel("Number of claims")
plt.title("Differences of Initial Claims Filed in California")
plt.tight_layout()
plt.show();

In [ ]:
p_max = 50
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6))
plot_acf(ytrain_CA_diff.dropna(), lags=p_max, ax=ax1, title="Sample ACF")
plot_pacf(ytrain_CA_diff.dropna(), lags=p_max, ax=ax2, title="Sample PACF")
plt.show();

### **Parameter Selection (Grid Search)**

In [ ]:
pmax, dmax, qmax = 8, 1, 6

def fit_arima(params):
    warnings.filterwarnings("ignore")
    p, d, q = params
    
    try:
        model = ARIMA(ytrain_CA, order=(p, d, q)).fit()
        forecast = model.get_forecast(steps=len(ytest_CA))
        yhat = forecast.predicted_mean
        yhat.index = ytest_CA.index
        rmse = np.sqrt(mean_squared_error(ytest_CA, yhat))
        
        return {
            "p": p,
            "d": d,
            "q": q,
            "AIC": model.aic,
            "BIC": model.bic,
            "RMSE": rmse
        }
    except Exception:
        return None

In [ ]:
cache_path_arima = "arima_grid_results_CAICLAIMS.csv"

param_grid = list(itertools.product(
    range(pmax + 1),
    range(dmax + 1),
    range(qmax + 1)
))

if os.path.exists(cache_path_arima):
    print(f"Loading cached ARIMA grid search results from {cache_path_arima}")
    results_arima_df = pd.read_csv(cache_path_arima)

else:
    print("Running ARIMA grid search...")

    results_arima = Parallel(n_jobs=-1, verbose=10)(
        delayed(fit_arima)(params) for params in param_grid
    )

    results_arima = [r for r in results_arima if r is not None]
    results_arima_df = pd.DataFrame(results_arima)

    if results_arima_df.empty:
        raise ValueError("ARIMA grid search failed: no models were successfully fit.")

    results_arima_df.to_csv(cache_path_arima, index=False)
    print(f"Saved ARIMA grid search results to {cache_path_arima}")

### **Best Models by Evaluation Metrics**

In [ ]:
best_aic = results_arima_df.loc[results_arima_df["AIC"].idxmin()]
best_bic = results_arima_df.loc[results_arima_df["BIC"].idxmin()]
best_rmse = results_arima_df.loc[results_arima_df["RMSE"].idxmin()]

print(
    f"Best model by AIC: ARIMA({int(best_aic['p'])}, "
    f"{int(best_aic['d'])}, {int(best_aic['q'])}), "
    f"AIC: {best_aic['AIC']:.3f}, ",
    f"BIC: {best_aic['BIC']:.3f}, ",
    f"RMSE: {best_aic['RMSE']:.3f}"
)
print(
    f"Best model by BIC: ARIMA({int(best_bic['p'])}, "
    f"{int(best_bic['d'])}, {int(best_bic['q'])}), "
    f"AIC: {best_bic['AIC']:.3f}, ",
    f"BIC: {best_bic['BIC']:.3f}, ",
    f"RMSE: {best_bic['RMSE']:.3f}"
)
print(
    f"Best model by RMSE: ARIMA({int(best_rmse['p'])}, "
    f"{int(best_rmse['d'])}, {int(best_rmse['q'])}), "
    f"AIC: {best_rmse['AIC']:.3f}, ",
    f"BIC: {best_rmse['BIC']:.3f}, ",
    f"RMSE: {best_rmse['RMSE']:.3f}"
)

In [ ]:
best_aic_model = ARIMA(ytrain_CA, order=(int(best_aic['p']), int(best_aic['d']), int(best_aic['q']))).fit()
print(best_aic_model.summary())

In [ ]:
best_bic_model = ARIMA(ytrain_CA, order=(int(best_bic['p']), int(best_bic['d']), int(best_bic['q']))).fit()
print(best_bic_model.summary())

In [ ]:
best_rmse_model = ARIMA(ytrain_CA, order=(int(best_rmse['p']), int(best_rmse['d']), int(best_rmse['q']))).fit()
print(best_rmse_model.summary())

### **Plotted ARIMA Predictions**

In [ ]:
best_aic_fcast = best_aic_model.get_prediction(start=train_end, end=len(y_CA) + test_horizon - 1)
best_aic_yhat = best_aic_fcast.predicted_mean
best_aic_rmse = np.sqrt(mean_squared_error(ytest_CA, best_aic_yhat.iloc[:len(ytest_CA)]))

best_bic_fcast = best_bic_model.get_prediction(start=train_end, end=len(y_CA) + test_horizon -1)
best_bic_yhat = best_bic_fcast.predicted_mean
best_bic_rmse = np.sqrt(mean_squared_error(ytest_CA, best_bic_yhat.iloc[:len(ytest_CA)]))

best_rmse_fcast = best_rmse_model.get_prediction(start=train_end, end=len(y_CA) + test_horizon - 1)
best_rmse_yhat = best_rmse_fcast.predicted_mean
best_rmse_rmse = np.sqrt(mean_squared_error(ytest_CA, best_rmse_yhat.iloc[:len(ytest_CA)]))


plt.figure(figsize=(10,6))
plt.plot(ttrain[(train_end - 150):], ytrain_CA[(train_end - 150):], color="steelblue", label="Training Data")
plt.plot(ttest, ytest_CA, color="red", label="Actual Future Values")

plt.plot(tpred, best_aic_yhat, color="blue", 
         label=f"Predicted Future Values\nBest AIC ARIMA({best_aic['p']:.0f}, {best_aic['d']:.0f}, {best_aic['q']:.0f}) Model")
plt.plot(tpred, best_bic_yhat, color="yellow", 
         label=f"Predicted Future Values\nBest BIC ARIMA({best_bic['p']:.0f}, {best_bic['d']:.0f}, {best_bic['q']:.0f}) Model")
plt.plot(tpred, best_rmse_yhat, color="orange", 
         label=f"Predicted Future Values\nBest RMSE ARIMA({best_rmse['p']:.0f}, {best_rmse['d']:.0f}, {best_rmse['q']:.0f}) Model")

plt.axvline(x=train_end, color="grey", linestyle="--", label="Train/Test Split")
plt.axvline(x=len(y_CA), color="black", linestyle="--", label="Dataset End (April 11, 2026)")

plt.xlabel("Time (Weekly: Feb 8, 1986 - )")
plt.ylabel("Number of claims")
plt.title("Initial Claims Filed in California")
plt.legend(); plt.show();

print(f"Root Mean Squared Error of Best AIC ARIMA({best_aic['p']:.0f}, {best_aic['d']:.0f}, {best_aic['q']:.0f}) Model: {best_aic_rmse}")
print(f"Root Mean Squared Error of Best BIC ARIMA({best_bic['p']:.0f}, {best_bic['d']:.0f}, {best_bic['q']:.0f}) Model: {best_bic_rmse}")
print(f"Root Mean Squared Error of Best RMSE ARIMA({best_rmse['p']:.0f}, {best_rmse['d']:.0f}, {best_rmse['q']:.0f}) Model: {best_rmse_rmse}")

## **2. SARIMAX with Fourier Frequencies, Before/After COVID Dummy**

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(ttrain, ytrain_CA, label="Training Data", color="steelblue")
plt.plot(ttest, ytest_CA, label="Testing Data", color="red")
plt.axvline(x=train_end, color="gray", linestyle="--", label="Train/Test Split")
plt.title("CAICLAIMS: Initial Claims Filed in California")
plt.xlabel("Time (Weekly: Feb 8, 1986 - April 11, 2026)")
plt.ylabel("Number of claims")
plt.legend(); plt.tight_layout(); plt.show();

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(ytrain_CA_diff.dropna())
plt.xlabel("Time (Weekly: Feb 8, 1986 - April 11, 2026)")
plt.ylabel("Number of claims")
plt.title("Differences of Initial Claims Filed in California")
plt.tight_layout()
plt.show();

### **Design Matrix Construction**

In [ ]:
def make_features(n, covid_start=1780):
    t = np.arange(n)
    X = pd.DataFrame(index=np.arange(n))

    # linear trend
    X["trend"] = t

    # yearly seasonality for weekly data
    for k in range(1, 4):  # 1st, 2nd, 3rd harmonic
        X[f"sin_{k}"] = np.sin(2 * np.pi * k * t / 52)
        X[f"cos_{k}"] = np.cos(2 * np.pi * k * t / 52)

    # COVID level-shift dummy
    X["covid_after"] = (t >= covid_start).astype(int)

    return X

### **SARIMAX Model Test**

In [ ]:
# Exog features through observed data only (for holdout RMSE / grid search)
X_CA = make_features(len(y_CA), covid_start=1780)

Xtrain_CA = X_CA.iloc[:train_end]
Xtest_CA = X_CA.iloc[train_end:]
Xtest_CA = Xtest_CA[Xtrain_CA.columns]

# Exog through observed data + future period (for extended future forecasts)
X_CA_extended = make_features(len(y_CA) + test_horizon, covid_start=1780)

Xtrain_CA_extended = X_CA_extended.iloc[:train_end]
Xtest_CA_extended = X_CA_extended.iloc[train_end : len(y_CA) + test_horizon]
Xtest_CA_extended = Xtest_CA_extended[Xtrain_CA_extended.columns]


# Best RMSE Baseline ARIMA(8, 1, 0) model with exogenous regressors
sarimax_model = SARIMAX(
    ytrain_CA,
    exog=Xtrain_CA_extended,
    order=(best_rmse['p'], best_rmse['d'], best_rmse['q']),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

print(sarimax_model.summary())

In [ ]:
# Forecast test period
sarimax_fcast = sarimax_model.get_forecast(steps=len(ytest_CA) + test_horizon, exog=Xtest_CA_extended)
sarimax_yhat = sarimax_fcast.predicted_mean
sarimax_yhat.index = tpred
sarimax_rmse = np.sqrt(mean_squared_error(ytest_CA, sarimax_yhat.iloc[:len(ytest_CA)]))


# Plot predictions against ARIMA baselines and actual values
plt.figure(figsize=(10, 6))
plt.plot(ttrain[(train_end - 150):], ytrain_CA.iloc[(train_end - 150):], label="Training Data", color="steelblue")
plt.plot(ttest, ytest_CA, label="Actual Future Values", color="red")

plt.plot(tpred, best_rmse_yhat, color="orange",
         label=f"Predicted Future Values\nBest RMSE ARIMA({best_rmse['p']:.0f}, {best_rmse['d']:.0f}, {best_rmse['q']:.0f}) Model")
plt.plot(tpred, sarimax_yhat, color="limegreen",
         label=f"Predicted Future Values\nBaseline SARIMAX({best_rmse['p']:.0f}, {best_rmse['d']:.0f}, {best_rmse['q']:.0f}) + Exog. Terms")

plt.axvline(x=train_end, color="gray", linestyle="--", label="Train/Test Split" )
plt.axvline(x=len(y_CA), color="black", linestyle="--", label="Dataset End (April 11, 2026)")

plt.title("Initial Claims Filed in California")
plt.xlabel("Time (Weekly: Feb 8, 1986 - )")
plt.ylabel("Number of claims")
plt.legend(); plt.tight_layout(); plt.show();


print(f"Root Mean Squared Error of Best RMSE ARMIA({best_rmse['p']:.0f}, {best_rmse['d']:.0f}, {best_rmse['q']:.0f}) Model: {best_rmse_rmse}")
print(f"Room Mean Squared Error of Baseline SARIMAX ({best_rmse['p']:.0f}, {best_rmse['d']:.0f}, {best_rmse['q']:.0f}) Model: {sarimax_rmse}")

### **Parameter Selection (Grid Search)**

In [ ]:
pmax, dmax, qmax = 8, 1, 6

def fit_sarimax_exog(params):
    warnings.filterwarnings("ignore")

    p, d, q = params

    try:
        model = SARIMAX(
            ytrain_CA,
            exog=Xtrain_CA,
            order=(p, d, q),
            enforce_stationarity=False,
            enforce_invertibility=False
        ).fit(disp=False)

        forecast = model.get_forecast(
            steps=len(ytest_CA),
            exog=Xtest_CA
        )

        y_hat = forecast.predicted_mean
        y_hat.index = ytest_CA.index

        rmse = np.sqrt(mean_squared_error(ytest_CA, y_hat))

        return {
            "p": p,
            "d": d,
            "q": q,
            "AIC": model.aic,
            "BIC": model.bic,
            "RMSE": rmse
        }

    except Exception:
        return None

In [ ]:
cache_path_sarimax = "sarimax_grid_results_CAICLAIMS.csv"

param_grid = list(itertools.product(
    range(pmax + 1),
    range(dmax + 1),
    range(qmax + 1)
))

if os.path.exists(cache_path_sarimax):
    print(f"Loading cached SARIMAX grid search results from {cache_path_sarimax}")
    results_sarimax_df = pd.read_csv(cache_path_sarimax)

else:
    print("Running SARIMAX grid search...")

    results_sarimax = Parallel(n_jobs=-1, verbose=10)(
        delayed(fit_sarimax_exog)(params) for params in param_grid
    )

    results_sarimax = [r for r in results_sarimax if r is not None]
    results_sarimax_df = pd.DataFrame(results_sarimax)

    if results_sarimax_df.empty:
        raise ValueError(
            "SARIMAX grid search failed: no models were successfully fit. "
            "Check that len(Xtest_CA) equals len(ytest_CA)."
        )

    results_sarimax_df.to_csv(cache_path_sarimax, index=False)
    print(f"Saved SARIMAX grid search results to {cache_path_sarimax}")

### **Best Models by Evaluation Metrics**

In [ ]:
best_aic_sarimax = results_sarimax_df.loc[results_sarimax_df["AIC"].idxmin()]
best_bic_sarimax = results_sarimax_df.loc[results_sarimax_df["BIC"].idxmin()]
best_rmse_sarimax = results_sarimax_df.loc[results_sarimax_df["RMSE"].idxmin()]

print(
    f"Best model by AIC: SARIMAX({int(best_aic_sarimax['p'])}, "
    f"{int(best_aic_sarimax['d'])}, {int(best_aic_sarimax['q'])}), "
    f"AIC: {best_aic_sarimax['AIC']:.3f}, "
    f"BIC: {best_aic_sarimax['BIC']:.3f}, "
    f"RMSE: {best_aic_sarimax['RMSE']:.3f}"
)

print(
    f"Best model by BIC: SARIMAX({int(best_bic_sarimax['p'])}, "
    f"{int(best_bic_sarimax['d'])}, {int(best_bic_sarimax['q'])}), "
    f"AIC: {best_bic_sarimax['AIC']:.3f}, "
    f"BIC: {best_bic_sarimax['BIC']:.3f}, "
    f"RMSE: {best_bic_sarimax['RMSE']:.3f}"
)

print(
    f"Best model by RMSE: SARIMAX({int(best_rmse_sarimax['p'])}, "
    f"{int(best_rmse_sarimax['d'])}, {int(best_rmse_sarimax['q'])}), "
    f"AIC: {best_rmse_sarimax['AIC']:.3f}, "
    f"BIC: {best_rmse_sarimax['BIC']:.3f}, "
    f"RMSE: {best_rmse_sarimax['RMSE']:.3f}"
)

In [ ]:
best_p = int(best_rmse_sarimax['p'])
best_d = int(best_rmse_sarimax['d'])
best_q = int(best_rmse_sarimax['q'])

best_rmse_sarimax_model = SARIMAX(
    ytrain_CA,
    exog=Xtrain_CA_extended,
    order=(best_p, best_d, best_q),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

print(best_rmse_sarimax_model.summary())

### **Plotted SARIMAX Predictions**

In [ ]:
best_rmse_sarimax_fcast = best_rmse_sarimax_model.get_forecast(steps=len(ytest_CA) + test_horizon, exog=Xtest_CA_extended)

best_rmse_sarimax_yhat = best_rmse_sarimax_fcast.predicted_mean
best_rmse_sarimax_yhat.index = tpred
best_sarimax_rmse = np.sqrt(mean_squared_error(ytest_CA, best_rmse_sarimax_yhat.iloc[:len(ytest_CA)]))


plt.figure(figsize=(10, 6))
plt.plot(ttrain[(train_end - 150):], ytrain_CA.iloc[(train_end - 150):], label="Training Data", color="steelblue")
plt.plot(ttest, ytest_CA, label="Actual Future Values", color="red")

plt.plot(tpred, best_rmse_yhat, color="orange",
         label=f"Predicted Future Values\nBest RMSE ARIMA({best_rmse['p']:.0f}, {best_rmse['d']:.0f}, {best_rmse['q']:.0f}) Model")
plt.plot(tpred, sarimax_yhat, color="limegreen",
         label=f"Predicted Future Values\nBaseline SARIMAX({best_rmse['p']:.0f}, {best_rmse['d']:.0f}, {best_rmse['q']:.0f}) + Exog. Terms")
plt.plot(tpred, best_rmse_sarimax_yhat, color="magenta",
         label=f"Predicted Future Values\nBest RMSE SARIMAX({best_p},{best_d},{best_q}) + Exog. Terms")

plt.axvline(x=train_end, color="gray", linestyle="--", label="Train/Test Split")
plt.axvline(x=len(y_CA), color="black", linestyle="--", label="Dataset End (April 11, 2026)")

plt.title("Initial Claims Filed in California")
plt.xlabel("Time (Weekly: Feb 8, 1986 - )")
plt.ylabel("Number of claims")
plt.legend(); plt.tight_layout(); plt.show();


print(f"Root Mean Squared Error of Best RMSE ARMIA({best_rmse['p']:.0f}, {best_rmse['d']:.0f}, {best_rmse['q']:.0f}) Model: {best_rmse_rmse}")
print(f"Room Mean Squared Error of Baseline SARIMAX ({best_rmse['p']:.0f}, {best_rmse['d']:.0f}, {best_rmse['q']:.0f}) Model: {sarimax_rmse}")
print(f"Root Mean Squared Error of Best RMSE SARIMAX({best_p:.0f}, {best_d:.0f}, {best_q:.0f}) Model: {best_sarimax_rmse}")

## **3. SARIMAX with Fourier Frequencies, During COVID Dummy**

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(ttrain, ytrain_CA, label="Training Data", color="steelblue")
plt.plot(ttest, ytest_CA, label="Testing Data", color="red")
plt.axvline(x=train_end, color="gray", linestyle="--", label="Train/Test Split")
plt.title("CAICLAIMS: Initial Claims Filed in California")
plt.xlabel("Time (Weekly: Feb 8, 1986 - April 11, 2026)")
plt.ylabel("Number of claims")
plt.legend(); plt.tight_layout(); plt.show();

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(ytrain_CA_diff.dropna())
plt.xlabel("Time (Weekly: Feb 8, 1986 - April 11, 2026)")
plt.ylabel("Number of claims")
plt.title("Differences of Initial Claims Filed in California")
plt.tight_layout()
plt.show();

### **Design Matrix Construction**

In [ ]:
def make_features_new(n, covid_start=1775, covid_end=1860, n_harmonics=3):
    t = np.arange(n)
    X = pd.DataFrame(index=np.arange(n))

    # Linear trend
    X["trend"] = t

    # Weekly data: yearly period is approximately 52 weeks
    for k in range(1, n_harmonics + 1):
        X[f"sin_{k}"] = np.sin(2 * np.pi * k * t / 52)
        X[f"cos_{k}"] = np.cos(2 * np.pi * k * t / 52)

    # COVID period indicator
    X["during_covid"] = ((t >= covid_start) & (t <= covid_end)).astype(int)

    return X

In [ ]:
# Exog features through observed data only (for holdout RMSE / grid search)
X_CA_new = make_features_new(n=len(y_CA), covid_start=1775, covid_end=1860, n_harmonics=3)

Xtrain_CA_new = X_CA_new.iloc[:train_end]
Xtest_CA_new = X_CA_new.iloc[train_end : len(y_CA) + test_horizon]
Xtest_CA_new = Xtest_CA_new[Xtrain_CA_new.columns]

# Exog through observed data + future period (for extended future forecasts)
X_CA_new_extended = make_features_new(n=len(y_CA) + test_horizon, covid_start=1775, covid_end=1860, n_harmonics=3)

Xtrain_CA_new_extended = X_CA_new_extended.iloc[:train_end]
Xtest_CA_new_extended = X_CA_new_extended.iloc[train_end : len(y_CA) + test_horizon]
Xtest_CA_new_extended = Xtest_CA_new_extended[Xtrain_CA_new_extended.columns]

### **Parameter Selection (Grid Search)**

In [ ]:
def rolling_cv_sarimax(
    y,
    X,
    order,
    initial_train_size=1500,
    horizon=52,
    step=26
):
    """
    Rolling-origin cross-validation for SARIMAX.

    At each split:
    - train on y[0:train_end]
    - forecast y[train_end:train_end+horizon]
    - compute RMSE
    """
    y = pd.Series(y).reset_index(drop=True)
    X = X.reset_index(drop=True)

    rows = []

    split_points = range(initial_train_size, len(y) - horizon + 1, step)

    for train_end in split_points:
        test_start = train_end
        test_end = train_end + horizon - 1

        y_train = y.iloc[:train_end]
        y_test = y.iloc[test_start:test_start + horizon]

        X_train = X.iloc[:train_end]
        X_test = X.iloc[test_start:test_start + horizon]

        # Make sure columns match exactly
        X_test = X_test[X_train.columns]

        try:
            model = SARIMAX(
                y_train,
                exog=X_train,
                order=order,
                enforce_stationarity=False,
                enforce_invertibility=False
            ).fit(disp=False)

            forecast = model.get_forecast(
                steps=horizon,
                exog=X_test
            )

            y_hat = forecast.predicted_mean
            y_hat.index = y_test.index

            rmse = np.sqrt(mean_squared_error(y_test, y_hat))

            rows.append({
                "train_end": train_end,
                "test_start": test_start,
                "test_end": test_end,
                "p": order[0],
                "d": order[1],
                "q": order[2],
                "RMSE": rmse,
                "AIC": model.aic,
                "BIC": model.bic,
                "status": "success"
            })

        except Exception as e:
            rows.append({
                "train_end": train_end,
                "test_start": test_start,
                "test_end": test_end,
                "p": order[0],
                "d": order[1],
                "q": order[2],
                "RMSE": np.nan,
                "AIC": np.nan,
                "BIC": np.nan,
                "status": "failed"
            })

    cv_results = pd.DataFrame(rows)

    mean_rmse = cv_results["RMSE"].mean(skipna=True)
    mean_aic = cv_results["AIC"].mean(skipna=True)
    mean_bic = cv_results["BIC"].mean(skipna=True)
    n_success = cv_results["RMSE"].notna().sum()

    return {
        "p": order[0],
        "d": order[1],
        "q": order[2],
        "CV_RMSE": mean_rmse,
        "CV_AIC": mean_aic,
        "CV_BIC": mean_bic,
        "n_success": n_success,
        "n_splits": len(cv_results)
    }

In [ ]:
cache_path = "sarimax_cv_grid_results_CAICLAIMS.csv"

pmax, dmax, qmax = 8, 1, 6

param_grid = list(itertools.product(
    range(pmax + 1),
    range(dmax + 1),
    range(qmax + 1)
))

initial_train_size = 1500
horizon = 52
step = 26


def evaluate_order_cv(order):
    return rolling_cv_sarimax(
        y=ytrain_CA,
        X=Xtrain_CA_new,
        order=order,
        initial_train_size=initial_train_size,
        horizon=horizon,
        step=step
    )


if os.path.exists(cache_path):
    print(f"Loading cached grid search results from {cache_path}")
    results_sarimax_new_df = pd.read_csv(cache_path)

else:
    print("Running SARIMAX rolling-CV grid search...")

    results_sarimax_new = Parallel(n_jobs=-1, verbose=10)(
        delayed(evaluate_order_cv)(order) for order in param_grid
    )

    results_sarimax_new_df = pd.DataFrame(results_sarimax_new)

    # Save cache
    results_sarimax_new_df.to_csv(cache_path, index=False)
    print(f"Saved grid search results to {cache_path}")

### **Best Models by Evaluation Metrics**

In [ ]:
# Clean and sort results
results_sarimax_new_df = results_sarimax_new_df.dropna(subset=["CV_RMSE"])
results_sarimax_new_df = results_sarimax_new_df.sort_values("CV_RMSE").reset_index(drop=True)

print("Top SARIMAX models by rolling-origin CV RMSE:")
display(results_sarimax_new_df.head(10))

In [ ]:
best_rmse_sarimax_new = results_sarimax_new_df.iloc[0]

best_p_new = int(best_rmse_sarimax_new['p'])
best_d_new = int(best_rmse_sarimax_new['d'])
best_q_new = int(best_rmse_sarimax_new['q'])

print(f"Best SARIMAX by rolling CV: SARIMAX({best_p_new}, {best_d_new}, {best_q_new})")
print(f"CV RMSE: {best_rmse_sarimax_new['CV_RMSE']:.3f}")
print(f"CV AIC: {best_rmse_sarimax_new['CV_AIC']:.3f}")
print(f"CV BIC: {best_rmse_sarimax_new['CV_BIC']:.3f}")
print(f"Successful CV splits: {int(best_rmse_sarimax_new['n_success'])} / {int(best_rmse_sarimax_new['n_splits'])}")

### **Final Holdout Evaluation**

In [ ]:
best_rmse_sarimax_model_new = SARIMAX(
    ytrain_CA,
    exog=Xtrain_CA_new_extended,
    order=(best_p_new, best_d_new, best_q_new),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

print(best_rmse_sarimax_model_new.summary())


best_rmse_sarimax_fcast_new = best_rmse_sarimax_model_new.get_forecast(
    steps=len(ytest_CA) + test_horizon,
    exog=Xtest_CA_new_extended
)

best_rmse_sarimax_yhat_new = best_rmse_sarimax_fcast_new.predicted_mean
best_rmse_sarimax_yhat_new.index = tpred
best_sarimax_rmse_new = np.sqrt(mean_squared_error(ytest_CA, best_rmse_sarimax_yhat_new.iloc[:len(ytest_CA)]))

print(f"\nFinal 2-year holdout RMSE for Best RMSE Rolling-CV SARIMAX({best_p_new:.0f}, {best_d_new:.0f}, {best_q_new:.0f}) Model: {best_sarimax_rmse_new}")

### **Plotted SARIMAX Predictions**

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(ttrain[(train_end - 150):], ytrain_CA.iloc[(train_end - 150):], label="Training Data", color="steelblue")
plt.plot(ttest, ytest_CA, label="Actual Future Values", color="red")

plt.plot(tpred, best_rmse_yhat, color="orange",
         label=f"Predicted Future Values\nBest RMSE ARIMA({best_rmse['p']:.0f}, {best_rmse['d']:.0f}, {best_rmse['q']:.0f}) Model")
plt.plot(tpred, best_rmse_sarimax_yhat, color="magenta",
         label=f"Predicted Future Values\nBest RMSE SARIMAX({best_p},{best_d},{best_q}) + Exog. Terms")
plt.plot(tpred, best_rmse_sarimax_yhat_new, color="skyblue",
         label=f"Predicted Future Values\nBest RMSE Rolling-CV SARIMAX({best_p_new},{best_d_new},{best_q_new}) + Exog. Terms")

plt.axvline(x=train_end, color="gray", linestyle="--", label="Train/Test Split")
plt.axvline(x=len(y_CA), color="black", linestyle="--", label="Dataset End (April 11, 2026)")

plt.title("Initial Claims Filed in California")
plt.xlabel("Time (Weekly: Feb 8, 1986 - )")
plt.ylabel("Number of claims")
plt.legend(); plt.tight_layout(); plt.show();


print(f"Root Mean Squared Error of Best RMSE ARMIA({best_rmse['p']:.0f}, {best_rmse['d']:.0f}, {best_rmse['q']:.0f}) Model: {best_rmse_rmse}")
print(f"Root Mean Squared Error of Best RMSE SARIMAX({best_p:.0f}, {best_d:.0f}, {best_q:.0f}) Model: {best_sarimax_rmse}")
print(f"Root Mean Squared Error of Best RMSE Rolling-CV SARIMAX({best_p_new:.0f}, {best_d_new:.0f}, {best_q_new:.0f}) Model: {best_sarimax_rmse_new}")